In [3]:
import pandas as pd
import kamping
import numpy as np
from kamping.data.dataset import MetaboliteProteinInteraction

%load_ext autoreload
%autoreload 2

In [62]:
mpi = pd.read_csv('../data/string_stitch/raw/9606.protein_chemical.links.v5.0.tsv', sep='\t')

# filter mpi with chemical not start with CIDs
mpi = mpi[~mpi['chemical'].str.startswith('CIDs')]

unique_compounds = mpi['chemical'].unique()
unique_compounds

array(['CIDm91758680', 'CIDm91758408', 'CIDm91758407', ...,
       'CIDm00000004', 'CIDm00000003', 'CIDm00000001'], dtype=object)

In [49]:
# remove compound start with CIDs
unique_compounds = [compound for compound in unique_compounds if not compound.startswith('CIDs')]
unique_compounds

['CIDm91758680',
 'CIDm91758408',
 'CIDm91758407',
 'CIDm91758406',
 'CIDm91758404',
 'CIDm91758403',
 'CIDm91758402',
 'CIDm91758401',
 'CIDm91758389',
 'CIDm91758271',
 'CIDm91757967',
 'CIDm91757966',
 'CIDm91757965',
 'CIDm91757964',
 'CIDm91757962',
 'CIDm91757960',
 'CIDm91757948',
 'CIDm91757947',
 'CIDm91757946',
 'CIDm91757945',
 'CIDm91757944',
 'CIDm91757941',
 'CIDm91757938',
 'CIDm91757937',
 'CIDm91757705',
 'CIDm91754990',
 'CIDm91754987',
 'CIDm91754984',
 'CIDm91754983',
 'CIDm91754982',
 'CIDm91754981',
 'CIDm91754979',
 'CIDm91754977',
 'CIDm91754976',
 'CIDm91754975',
 'CIDm91754974',
 'CIDm91754972',
 'CIDm91754971',
 'CIDm91754969',
 'CIDm91754964',
 'CIDm91754961',
 'CIDm91754958',
 'CIDm91754956',
 'CIDm91754955',
 'CIDm91754953',
 'CIDm91754952',
 'CIDm91754951',
 'CIDm91754946',
 'CIDm91754711',
 'CIDm91754604',
 'CIDm91754603',
 'CIDm91754602',
 'CIDm91754600',
 'CIDm91754581',
 'CIDm91754554',
 'CIDm91754243',
 'CIDm91754234',
 'CIDm91754232',
 'CIDm91754231

In [69]:
import pandas as pd

# Initialize an empty DataFrame to store the filtered data
filtered_smiles = pd.DataFrame()

# Define the chunk size
chunk_size = 1000000  # Adjust based on your memory capacity

# Read the first TSV file in chunks
for chunk in pd.read_csv('../data/string_stitch/raw/chemicals.v5.0.tsv', sep='\t', chunksize=chunk_size):
    # Filter the chunk based on unique_compounds
    filtered_chunk = chunk[chunk['chemical'].isin(unique_compounds)]
    # Append the filtered chunk to the filtered_smiles DataFrame
    filtered_smiles = pd.concat([filtered_smiles, filtered_chunk], ignore_index=True)

# Now filtered_smiles contains the filtered data

In [70]:
# # Save the filtered data to a new TSV file
filtered_smiles.to_csv('../data/string_stitch/raw/filtered_output.tsv', sep='\t', index=False)

In [71]:
# import the smile strings
smiles = pd.read_csv('../data/string_stitch/raw/filtered_output.tsv', sep='\t', usecols=['chemical', 'SMILES_string'], index_col='chemical')
smiles

,SMILES_string
chemical,
CIDm00000001,CC(=O)OC(CC(=O)[O-])C[N+](C)(C)C
CIDm00000003,C1=CC(C(C(=C1)C(=O)O)O)O
CIDm00000004,CC(CN)O
CIDm00000005,C(C(=O)COP(=O)(O)O)N
CIDm00000006,C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl
...,...
CIDm91758404,C1CCC(CC1)COC2=CC=CC(=C2)C(CCO)O
CIDm91758406,C1COCCC1NC2=NC=C3CCN(CC3=N2)S(=O)(=O)CCS
CIDm91758407,CNC(=O)C1=CC=CC=C1NC2=CC(=NC=C2Cl)NC3CCOCC3


In [72]:
from rdkit import Chem

mols = []

for compound in unique_compounds:
    # check if the compound exists in the smiles DataFrame
    if compound in smiles.index:
        # convert the SMILES string to a RDKit molecule object
        mols.append(Chem.MolFromSmiles(smiles.loc[compound, 'SMILES_string']))
    else:
        # if the compound does not exist, append None
        print(f'{compound} not found')
        mols.append(None)
    # create a pd.DataFrame
ROMols = pd.DataFrame({'id': list(unique_compounds), 'ROMol': mols})
# remove the None values
ROMols = ROMols.dropna()

CIDm73330332 not found
CIDm70678717 not found
CIDm44237211 not found
CIDm44237133 not found
CIDm25098796 not found
CIDm05189703 not found
CIDm03080603 not found
CIDm00002828 not found


In [75]:
compound_embeddings = kamping.get_mol_embeddings_from_dataframe(ROMols, transformer='morgan', dtype=np.int8)

'
                    total 0 Invalid rows with "None" in the ROMol column


In [76]:
# save the embeddings using h5py
import h5py
with h5py.File('../data/string_stitch/raw/compound_embeddings.h5', 'w') as f:
    for key, value in compound_embeddings.items():
        f.create_dataset(key, data=value)

In [77]:
compound_embeddings

{'CIDm91758680': array([0, 0, 0, ..., 0, 0, 1], dtype=int8),
 'CIDm91758408': array([0, 0, 0, ..., 0, 0, 0], dtype=int8),
 'CIDm91758407': array([0, 0, 0, ..., 0, 1, 0], dtype=int8),
 'CIDm91758406': array([0, 0, 0, ..., 0, 0, 0], dtype=int8),
 'CIDm91758404': array([0, 1, 1, ..., 0, 0, 0], dtype=int8),
 'CIDm91758403': array([0, 1, 0, ..., 0, 0, 0], dtype=int8),
 'CIDm91758402': array([0, 1, 0, ..., 0, 0, 0], dtype=int8),
 'CIDm91758401': array([0, 1, 0, ..., 0, 0, 0], dtype=int8),
 'CIDm91758389': array([0, 0, 0, ..., 0, 0, 0], dtype=int8),
 'CIDm91758271': array([0, 0, 0, ..., 0, 0, 0], dtype=int8),
 'CIDm91757967': array([0, 0, 0, ..., 0, 0, 0], dtype=int8),
 'CIDm91757966': array([0, 0, 0, ..., 0, 0, 0], dtype=int8),
 'CIDm91757965': array([0, 1, 1, ..., 0, 0, 0], dtype=int8),
 'CIDm91757964': array([0, 0, 0, ..., 0, 0, 0], dtype=int8),
 'CIDm91757962': array([0, 0, 0, ..., 0, 0, 0], dtype=int8),
 'CIDm91757960': array([0, 0, 0, ..., 0, 0, 0], dtype=int8),
 'CIDm91757948': array([

In [15]:
# load the dictionary of embeddings from the h5 file
with h5py.File("../data/string_stitch/raw/compound_embeddings.h5", "r") as f:
    # Initialize an empty dictionary
    compound_embeddings = {}

    # Iterate over the items in the file
    for key in f.keys():
        # Read each dataset into the dictionary
        compound_embeddings[key] = f[key][...]

In [94]:
compound_embeddings

{'CIDm91758680': array([0, 0, 0, ..., 0, 0, 1], dtype=int8),
 'CIDm91758408': array([0, 0, 0, ..., 0, 0, 0], dtype=int8),
 'CIDm91758407': array([0, 0, 0, ..., 0, 1, 0], dtype=int8),
 'CIDm91758406': array([0, 0, 0, ..., 0, 0, 0], dtype=int8),
 'CIDm91758404': array([0, 1, 1, ..., 0, 0, 0], dtype=int8),
 'CIDm91758403': array([0, 1, 0, ..., 0, 0, 0], dtype=int8),
 'CIDm91758402': array([0, 1, 0, ..., 0, 0, 0], dtype=int8),
 'CIDm91758401': array([0, 1, 0, ..., 0, 0, 0], dtype=int8),
 'CIDm91758389': array([0, 0, 0, ..., 0, 0, 0], dtype=int8),
 'CIDm91758271': array([0, 0, 0, ..., 0, 0, 0], dtype=int8),
 'CIDm91757967': array([0, 0, 0, ..., 0, 0, 0], dtype=int8),
 'CIDm91757966': array([0, 0, 0, ..., 0, 0, 0], dtype=int8),
 'CIDm91757965': array([0, 1, 1, ..., 0, 0, 0], dtype=int8),
 'CIDm91757964': array([0, 0, 0, ..., 0, 0, 0], dtype=int8),
 'CIDm91757962': array([0, 0, 0, ..., 0, 0, 0], dtype=int8),
 'CIDm91757960': array([0, 0, 0, ..., 0, 0, 0], dtype=int8),
 'CIDm91757948': array([

In [79]:
import numpy as np
from sklearn.decomposition import PCA

# Extract the embeddings and keys
keys = list(compound_embeddings.keys())
embeddings = np.array(list(compound_embeddings.values()))

# Apply PCA to reduce to 128 dimensions
pca = PCA(n_components=128)
reduced_embeddings = pca.fit_transform(embeddings)

# Create a new dictionary with the reduced embeddings
reduced_mol_embeddings = {keys[i]: reduced_embeddings[i] for i in range(len(keys))}

# Print the reduced embeddings dictionary
# print(reduced_mol_embeddings)

In [80]:
# convert reduced_mol_embeddings to float32
for key, value in reduced_mol_embeddings.items():
    reduced_mol_embeddings[key] = value.astype(np.float32)

# save to h5 file
with h5py.File('../data/string_stitch/raw/compound_embeddings_PCA.h5', 'w') as f:
    for key, value in reduced_mol_embeddings.items():
        f.create_dataset(key, data=value)

# Convert string to Uniprot ID

In [82]:
uniprot_df = pd.read_table('/Users/cgu3/Documents/Grape-Pi/data/miscellaneous/uniprotkb_proteome_UP000005640_AND_revi_2023_10_05.tsv', sep='\t')
uniprot_df = uniprot_df[['STRING', 'Entry']]
uniprot_df.dropna(subset=['STRING'], inplace=True)
uniprot_df['STRING'] = uniprot_df['STRING'].str.replace(';', '')
uniprot_df

,STRING,Entry
2,9606.ENSP00000482829,A0A0B4J2F2
7,9606.ENSP00000377112,A0AV02
8,9606.ENSP00000371212,A0AV96
9,9606.ENSP00000419279,A0AVF1
10,9606.ENSP00000372394,A0AVI4
...,...,...
20371,9606.ENSP00000429608,Q96PS1
20373,9606.ENSP00000402355,Q96T59
20383,9606.ENSP00000359558,Q9H1L0
20388,9606.ENSP00000455079,Q9H693


In [83]:
mpi['protein'] = kamping.utils.mapping(mpi['protein'], uniprot_df, na_rm=False)
# drop the rows with missing values
mpi.dropna(subset=['protein'], inplace=True)

In [84]:
mpi.to_csv('../data/string_stitch/raw/9606.protein_chemical_translated.links.v5.0.tsv', sep='\t', index=False)

In [4]:
# load ../data/string_stitch/raw/9606.protein_chemical_translated.links.v5.0.ts
mpi = pd.read_csv('../data/string_stitch/raw/9606.protein_chemical_translated.links.v5.0.tsv', sep='\t')
mpi

,chemical,protein,combined_score
0,CIDm91758680,Q9BZR9,154
1,CIDm91758408,Q6NTF9,225
2,CIDm91758408,Q12774,178
3,CIDm91758408,Q9Y3P4,225
4,CIDm91758408,O95140,162
...,...,...,...
5949655,CIDm00000001,Q00059,150
5949656,CIDm00000001,Q9NTX5,279
5949657,CIDm00000001,P40925,311
5949658,CIDm00000001,O14727,322


In [11]:
mpi[mpi['combined_score'] > 900].to_csv('../data/string_stitch/raw/9606.protein_chemical_translated.filtered.links.v5.0.tsv', sep='\t', index=False)

# Process metabolite-metabolite interaction data

In [88]:
mmi = pd.read_csv('../data/string_stitch/raw/chemical_chemical.links.detailed.v5.0.tsv', sep='\t')
mmi

,chemical1,chemical2,similarity,experimental,database,textmining,combined_score
0,CIDm00024759,CIDs00024759,0,0,900,0,900
1,CIDs91758695,CIDs00107694,0,0,0,230,230
2,CIDs91758695,CIDs11013287,0,0,0,230,230
3,CIDs91758695,CIDs11980957,0,0,0,328,328
4,CIDs91758695,CIDs00013078,0,0,0,162,162
...,...,...,...,...,...,...,...
17705813,CIDm00000001,CIDm87081431,753,0,0,173,173
17705814,CIDm00000001,CIDm90857042,0,0,0,218,218
17705815,CIDm00000001,CIDm91056687,0,0,0,294,294
17705816,CIDm00000001,CIDm91213096,0,0,0,357,357


In [89]:
# filter row with chemical1 start with CIDs or chemical2 start with CIDs
mmi = mmi[~mmi['chemical1'].str.startswith('CIDs')]
mmi = mmi[~mmi['chemical2'].str.startswith('CIDs')]
mmi

,chemical1,chemical2,similarity,experimental,database,textmining,combined_score
9665846,CIDm91758680,CIDm00000066,0,0,0,230,230
9665847,CIDm91758680,CIDm00000124,750,0,0,274,274
9665848,CIDm91758680,CIDm00000136,0,0,0,170,170
9665849,CIDm91758680,CIDm00001145,0,0,0,179,179
9665850,CIDm91758680,CIDm00006914,0,0,0,215,215
...,...,...,...,...,...,...,...
17705812,CIDm00000001,CIDm87076413,500,0,0,176,176
17705813,CIDm00000001,CIDm87081431,753,0,0,173,173
17705814,CIDm00000001,CIDm90857042,0,0,0,218,218
17705815,CIDm00000001,CIDm91056687,0,0,0,294,294


# Use dataset to create a graph from the data

In [15]:
data = MetaboliteProteinInteraction(root='../data/string_stitch')

Processing...
/Users/cgu3/Documents/experiments/KAMPING/kamping/data/dataset.py:67: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:281.)
  data['gene'].x = torch.tensor([protein_embeddings[node] for node in proteins if node in protein_embeddings], dtype=torch.float)
Done!
/Users/cgu3/Documents/experiments/KAMPING/kamping/data/dataset.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be 

In [97]:
data[0]

HeteroData(
  gene={ x=[16752, 1024] },
  compound={ x=[343286, 128] },
  (gene, to, gene)={ edge_index=[2, 10605120] },
  (compound, to, gene)={ edge_index=[2, 5596399] }
)

In [100]:
import pickle

# save the data as pickle file
with open('../data/pyg_graph_with_string_stitch_900_metabolite', 'wb') as f:
    pickle.dump(data[0], f)